# Schedule-Component Ablations and Daily-Baseline Schedule Tests

Tests the trigger scheduler's two separate design choices against controls: the schedule-component ablations (crossover and random 2x2 grids over training-date and trading-date restriction), the daily-baseline schedule tests (a fixed daily-trained agent evaluated under restricted trading dates), and a cross-distribution generalization check on the crossover-trained models.

## Cross-Distribution Generalization Test

**What happens when a model is evaluated on trading dates different from what it trained on?**

The main pipeline trains and evaluates each model on matching training and trading dates:
a crossover-trained model (`flt_{fam}`, `flt_{fam}_trg`) is only ever evaluated on crossover
trading dates; a daily-trained model (`{fam}`) is only ever evaluated on daily trading dates.
This notebook cross-evaluates the already-trained models from `experiment_1` (read-only, no
retraining here) on the *other* schedule's trading dates, for the `mvx2` and `bollx` families:

1. `flt_{fam}` (crossover-trained, indicator-and-crossover representation) → evaluated on
   **daily** trading dates (including days it never saw during training).
2. `flt_{fam}_trg` (crossover-trained, crossover representation) → evaluated on **daily**
   trading dates.
3. `{fam}` (daily-trained) → evaluated **only** on crossover trading dates (the reverse
   direction).

New results are cached separately under `experiment_2`, distinct from `experiment_1`'s
matching-schedule reference numbers loaded read-only below for comparison.

In [1]:
%matplotlib inline
import os, sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import A2C, PPO, SAC
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
sys.path.insert(0, str(Path(os.getcwd())))
from utils.config import SEEDS, WINDOWS
from utils.common import (
    make_window_data, get_indicators, make_env_kwargs,
    run_prediction_vecnorm, bootstrap_ci, compare_placebo,
    run_prediction_vecnorm_reconstructed, actions_to_positions,
    control_model_run_name, manifest_row_lookup, load_frozen_target_dates,
)
from utils.tables import compute_metrics

In [ ]:
FINRL_PATH = Path(os.environ['FINRL_PATH'])
MODELS_DIR = FINRL_PATH / 'models' / 'dow' / 'experiment_1'   # reuse trained models, no retraining

FAMILIES = {
    'mvx2':  {'full': 'mvx2',  'flt': 'flt_mvx2',  'flt_trg': 'flt_mvx2_trg',  'full_trg': 'mvx2_trg'},
    'bollx': {'full': 'bollx', 'flt': 'flt_bollx', 'flt_trg': 'flt_bollx_trg', 'full_trg': 'bollx_trg'},
}
ALL_DATASETS = sorted({d for fam in FAMILIES.values() for d in fam.values()} | {'all'})

ALGOS = {'a2c': A2C, 'ppo': PPO, 'sac': SAC}
WINDOWS_TO_RUN = ['w1', 'w4', 'w3']
DEVICE = 'cpu'

NATIVE_CACHE_DIR      = FINRL_PATH / 'results' / 'dow' / 'experiment_1'
NATIVE_BACKTEST_CACHE = NATIVE_CACHE_DIR / 'backtest_cache_mv2boll.pkl'   # read-only reference

CACHE_DIR      = FINRL_PATH / 'results' / 'dow' / 'experiment_2'  # this notebook's own output
BACKTEST_CACHE = CACHE_DIR / 'backtest_cache_mv2boll.pkl'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RERUN = False   # set True to redo the full inference sweep even if BACKTEST_CACHE exists

AGG_METHOD = 'mean_metrics'  

print(f'Models (reused, read-only): {MODELS_DIR}')
print(f'Native reference cache:     {NATIVE_BACKTEST_CACHE}  (exists={NATIVE_BACKTEST_CACHE.exists()})')
print(f'New cross-eval cache:       {BACKTEST_CACHE}  (exists={BACKTEST_CACHE.exists()})')

Models (reused, read-only): /home/main/Programming/finrl/models/dow/experiment_1
Native reference cache:     /home/main/Programming/finrl/results/dow/experiment_1/backtest_cache_mv2boll.pkl  (exists=True)
New cross-eval cache:       /home/main/Programming/finrl/results/dow/experiment_2/backtest_cache_mv2boll.pkl  (exists=True)


In [3]:
# Load-or-compute guard: if a complete cache already exists on disk, load it and skip
# the three expensive inference sweeps below (cells building cross_results/recon_results/
# de_results) entirely -- same FORCE_RERUN/_cache_loaded pattern dow_3_backtest.ipynb
# already uses. Previously this notebook had no load path at all, so BACKTEST_CACHE was
# written on every run but never read back -- every run redid all ~270+ rollouts from
# scratch regardless of whether a good cache already existed.
if not FORCE_RERUN and BACKTEST_CACHE.exists():
    with open(BACKTEST_CACHE, 'rb') as f:
        cross_results = pickle.load(f)['results']
    _cache_loaded = True
    print(f'✓ Loaded from cache: {BACKTEST_CACHE}')
    print(f'  Windows: {list(cross_results.keys())}')
else:
    _cache_loaded = False
    if BACKTEST_CACHE.exists():
        print('[FORCE_RERUN=True] -- ignoring existing cache.')
    else:
        print('No cache found -- will compute from scratch.')

✓ Loaded from cache: /home/main/Programming/finrl/results/dow/experiment_2/backtest_cache_mv2boll.pkl
  Windows: ['w1', 'w4', 'w3']


In [4]:
# Load NATIVE (matching-distribution) reference results — read-only, for comparison
with open(NATIVE_BACKTEST_CACHE, 'rb') as f:
    native_cache = pickle.load(f)
native_results = native_cache['results']

def aggregate_seeds(results):
    agg = {}
    for window, ds_dict in results.items():
        agg[window] = {}
        for ds, algo_dict in ds_dict.items():
            agg[window][ds] = {}
            for algo, seed_list in algo_dict.items():
                if not seed_list:
                    continue
                mat = pd.concat(seed_list, axis=1)
                agg[window][ds][algo] = {
                    'mean': mat.mean(axis=1),
                    'std': mat.std(axis=1),
                    'n': len(seed_list),
                }
    return agg

native_agg = aggregate_seeds(native_results)
print(f'✓ Native results loaded: {len(native_results)} windows')

# Load window data (train tags needed for model paths + test dfs needed for eval)
data = make_window_data(ALL_DATASETS, windows=WINDOWS_TO_RUN, cvtd=False, splits=['train', 'test'])
print(f'✓ Loaded data: {len(data)} windows')
print()
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        n_full = len(data[w][names['full']]['test']['df'])
        n_flt  = len(data[w][names['flt']]['test']['df'])
        print(f'  {w} {fam:6s}: full test rows={n_full:6d}  filtered test rows={n_flt:6d}  ({n_flt/n_full:.1%} kept)')

✓ Native results loaded: 3 windows


✓ Loaded data: 3 windows

  w1 mvx2  : full test rows=  6390  filtered test rows=  2460  (38.5% kept)
  w1 bollx : full test rows=  6390  filtered test rows=  3600  (56.3% kept)
  w4 mvx2  : full test rows=  7530  filtered test rows=  3330  (44.2% kept)
  w4 bollx : full test rows=  7530  filtered test rows=  4380  (58.2% kept)
  w3 mvx2  : full test rows=  7500  filtered test rows=  3150  (42.0% kept)
  w3 bollx : full test rows=  7500  filtered test rows=  5100  (68.0% kept)


In [5]:
# Model availability check — run before anything expensive
missing_models = []
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        for role in ('full', 'flt', 'flt_trg'):
            ds = names[role]
            train_tag = data[w][ds]['train']['tag']
            for algo in ALGOS:
                for seed in SEEDS:
                    p = MODELS_DIR / f'{train_tag}_{algo}_s{seed}.zip'
                    if not p.exists():
                        missing_models.append(str(p))

print(f'Missing models: {len(missing_models)}')
for m in missing_models[:10]:
    print(' ', m)

Missing models: 0


## Gap-aware equity reconstruction

`compute_metrics` treats every row of an equity curve as one consecutive trading day. That
holds for daily trading dates, but not for crossover trading dates, where consecutive
retained rows can be several calendar days apart. Treating a multi-day gap as one day
inflates annualized Sharpe/CAGR/Volatility by roughly the square root of the gap and
truncates the measured period early.

Between two crossover trading dates the agent holds a fixed position, so the account's
true mark-to-market value on the skipped days is exactly reconstructable: carry the
holdings forward from `state_memory` and reprice against the real daily close from the
full (unfiltered) dataset. Validated against the agent's own recorded account values at
every crossover trading date. `reconstruct_daily_curve` / `run_prediction_vecnorm_reconstructed`
in `utils/common.py` implement this and are shared with `dow_3_backtest.ipynb` and
`dow_5_controls.ipynb`.

Only the three conditions below actually need reconstruction, since their evaluation
dates are the crossover schedule:
- `flt_{fam}` evaluated on its own crossover trading dates — native specialist
- `flt_{fam}_trg` evaluated on its own crossover trading dates — native specialist_trg
- `{fam}` evaluated on crossover trading dates — cross generalist→filtered

The other three conditions (`{fam}` on its own daily trading dates, `flt_{fam}`/`flt_{fam}_trg`
on daily trading dates) are already exact daily curves and reuse the values computed above.

In [6]:
def run_eval_reconstructed(model_ds, eval_df, window, algo_name, algo_cls, seed,
                            full_price_wide, full_dates):
    """
    Like run_cross_eval, but for evaluations on a FILTERED (gappy) eval_df -- thin
    wrapper around utils.common.run_prediction_vecnorm_reconstructed (moved there from this
    notebook, and now shared with dow_3_backtest.ipynb / dow_5_controls.ipynb instead of
    being duplicated three ways). Also inherits a real fix this notebook's own copy
    never had: the reconstructed curve is anchored to the full calendar's TRUE start
    date, not just the first trigger/decision date -- see
    utils.common.reconstruct_daily_curve's docstring. positions_df (per-ticker holdings) is
    discarded here; no caller in this notebook needs it.
    """
    train_tag = data[window][model_ds]['train']['tag']
    run_name = f'{train_tag}_{algo_name}_s{seed}'
    model_path = MODELS_DIR / f'{run_name}.zip'
    vecnorm_path = MODELS_DIR / f'{run_name}_vecnorm.pkl'
    if not (model_path.exists() and vecnorm_path.exists()):
        return None

    indicators = get_indicators(model_ds, df=eval_df)
    curve, _positions_df = run_prediction_vecnorm_reconstructed(
        model_path, vecnorm_path, algo_cls, eval_df, indicators,
        full_price_wide, full_dates, device=DEVICE,
    )
    return curve.rename(f's{seed}')

In [7]:
def run_cross_eval(model_ds, eval_df, window, algo_name, algo_cls, seed):
    """
    Load the model trained on `model_ds` (window, algo, seed) and run it against
    `eval_df` — which may come from a DIFFERENT dataset than `model_ds`. Generalizes
    this notebook's random-subsample placebo pattern to a full alternate test set.
    """
    train_tag = data[window][model_ds]['train']['tag']
    run_name = f'{train_tag}_{algo_name}_s{seed}'
    model_path = MODELS_DIR / f'{run_name}.zip'
    vecnorm_path = MODELS_DIR / f'{run_name}_vecnorm.pkl'
    if not (model_path.exists() and vecnorm_path.exists()):
        return None

    model = algo_cls.load(str(model_path), device=DEVICE)
    indicators = get_indicators(model_ds, df=eval_df)
    env_kwargs = make_env_kwargs(eval_df, indicators)

    df_eval = eval_df.copy()
    df_eval['day'] = pd.factorize(df_eval['date'])[0]
    df_indexed = df_eval.set_index('day')

    test_env = StockTradingEnv(df=df_indexed, **env_kwargs)
    df_av, _ = run_prediction_vecnorm(model, test_env, vecnorm_path)
    return df_av.set_index('date')['account_value'].rename(f's{seed}')

In [8]:
# Single dry-run probe — cheap sanity check before the full sweep
_probe_curve = run_cross_eval(
    model_ds='flt_mvx2',
    eval_df=data['w1']['mvx2']['test']['df'],   # FULL test set, flt_mvx2-trained model
    window='w1', algo_name='a2c', algo_cls=A2C, seed=42,
)
print('cross  flt_mvx2 → full test  (w1/a2c/seed42):', compute_metrics(_probe_curve))
print('native flt_mvx2 (w1/a2c/seed42, own filtered test):',
      compute_metrics(native_results['w1']['flt_mvx2']['a2c'][0]))

Stock Dimension: 30, State Space: 181


cross  flt_mvx2 → full test  (w1/a2c/seed42): {'Return%': np.float64(30.17), 'CAGR': np.float64(36.6), 'Sharpe': 1.015, 'Sortino': 1.393, 'Calmar': 1.356, 'MaxDD%': -27.14, 'Volatility%': np.float64(38.1)}
native flt_mvx2 (w1/a2c/seed42, own filtered test): {'Return%': np.float64(41.64), 'CAGR': np.float64(50.97), 'Sharpe': 1.396, 'Sortino': 2.016, 'Calmar': 2.445, 'MaxDD%': -20.96, 'Volatility%': np.float64(33.74)}


In [9]:
# Full cross-evaluation sweep
# 3 windows x 2 families x 2 conditions x 3 algos x 5 seeds = 180 inference rollouts
# (pure inference, no training — expect low single-digit minutes on CPU)
# The third historical condition (full-trained -> on_filtered) is intentionally NOT
# here: its eval set is filtered, so it needs gap-aware reconstruction -- it's computed
# once, correctly, in the reconstruction sweep below. Computing it here too would just
# be immediately overwritten and wasted (this used to be a real 1/3-of-this-sweep waste).
# CROSS_CONDITIONS stays defined even on a cache hit -- the final save's metadata
# references it regardless of whether this sweep actually ran.
CROSS_CONDITIONS = [
    # (label_suffix, model_role, eval_role)
    ('on_full',     'flt',     'full'),   # flt-trained      → full test
    ('on_full',     'flt_trg', 'full'),   # flt_trg-trained  → full test
]

if not _cache_loaded:
    cross_results = {}
    for window in WINDOWS_TO_RUN:
        cross_results[window] = {}
        for fam, names in FAMILIES.items():
            for suffix, model_role, eval_role in CROSS_CONDITIONS:
                model_ds = names[model_role]
                eval_df  = data[window][names[eval_role]]['test']['df']
                label    = f'{model_ds}__{suffix}'

                cross_results[window][label] = {}
                for algo_name, algo_cls in ALGOS.items():
                    seed_curves = []
                    for seed in SEEDS:
                        s = run_cross_eval(model_ds, eval_df, window, algo_name, algo_cls, seed)
                        if s is not None:
                            seed_curves.append(s)
                    cross_results[window][label][algo_name] = seed_curves
                    print(f'  {window} {label:28s} {algo_name}: {len(seed_curves)}/{len(SEEDS)} seeds')

    print('✓ Cross-evaluation sweep complete')
else:
    print('[cache loaded] Skipping cross-evaluation sweep.')

[cache loaded] Skipping cross-evaluation sweep.


In [10]:
# Unified reconstruction sweep: every condition whose EVALUATION dataframe is filtered
# needs gap-aware reconstruction (run_eval_reconstructed) -- native flt/flt_trg
# specialists, the full-trained generalist forced onto filtered dates (the corrected
# version of the condition CROSS_CONDITIONS above deliberately skips), and the DE
# condition (daily-trained boolean-only-state model forced onto filtered dates --
# formerly a separate sweep keyed by TRG_FAMILIES, now just a 4th recon_condition per
# family via the 'full_trg' role added to FAMILIES; ALL_DATASETS already pulls in
# mvx2_trg/bollx_trg automatically, so no separate extra_data load is needed either).
# Ends by folding into cross_results and doing the ONE disk save for this notebook.
if not _cache_loaded:
    recon_results = {}
    for w in WINDOWS_TO_RUN:
        recon_results[w] = {}
        for fam, names in FAMILIES.items():
            full_test_df = data[w][names['full']]['test']['df']
            tickers = sorted(full_test_df['tic'].unique())
            full_price_wide = full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
            full_price_wide.index = pd.to_datetime(full_price_wide.index)
            full_dates = full_price_wide.index
            flt_test_df = data[w][names['flt']]['test']['df']

            recon_conditions = [
                (names['flt'],                        names['flt'],     flt_test_df),
                (names['flt_trg'],                    names['flt_trg'], data[w][names['flt_trg']]['test']['df']),
                (f"{names['full']}__on_filtered",     names['full'],     flt_test_df),
                (f"{names['full_trg']}__on_filtered", names['full_trg'], flt_test_df),
            ]

            for label, model_ds, eval_df in recon_conditions:
                recon_results[w][label] = {}
                for algo_name, algo_cls in ALGOS.items():
                    seed_curves = []
                    for seed in SEEDS:
                        c = run_eval_reconstructed(model_ds, eval_df, w, algo_name, algo_cls, seed,
                                                    full_price_wide, full_dates)
                        if c is not None:
                            seed_curves.append(c)
                    recon_results[w][label][algo_name] = seed_curves
                    print(f'  {w} {label:28s} {algo_name}: {len(seed_curves)}/{len(SEEDS)} seeds (reconstructed)')

    print('✓ Reconstruction sweep complete (native specialists, generalist-on-filtered, DE)')

    for w in WINDOWS_TO_RUN:
        cross_results[w].update(recon_results[w])
    print('✓ Corrected + DE values folded into cross_results.')

    with open(BACKTEST_CACHE, 'wb') as f:
        pickle.dump({'results': cross_results, 'families': FAMILIES, 'conditions': CROSS_CONDITIONS,
                     'reconstructed_labels': [label for w in recon_results for label in recon_results[w]]}, f)
    print(f'✓ Saved: {BACKTEST_CACHE}')
else:
    print('[cache loaded] Skipping reconstruction sweep.')

[cache loaded] Skipping reconstruction sweep.


In [11]:
# Merge native + cross results into one agg dict, reuse tables.make_metrics_table as-is.
# cross_results is already fully corrected by this point (recon_results folded in above,
# or loaded pre-corrected from cache) -- no separate recon_agg merge needed anymore.
from utils.tables import make_metrics_table
    
SAVE = False
SAVE_DIR = "tables_mv2boll_2x2"
cross_agg = aggregate_seeds(cross_results)

combined_agg = {w: dict(native_agg.get(w, {})) for w in WINDOWS_TO_RUN}
for w in WINDOWS_TO_RUN:
    combined_agg[w].update(cross_agg.get(w, {}))

# Raw per-seed curves, merged the same way as combined_agg -- passed as results= below so
# make_metrics_table also computes Return%_mean/Sharpe_mean/MaxDD%_mean (metric computed
# on each seed curve, then averaged) alongside the plain Return%/Sharpe/MaxDD% columns
# (metric computed once on the seed-AVERAGED curve, via aggregate_seeds' 'mean' curve).
# The two differ: averaging curves first mechanically cancels cross-seed volatility the
# way averaging independent portfolios does, so the plain columns run optimistic on
# Sharpe and shallow on MaxDD. Use the _mean columns for any comparison ACROSS
# conditions -- delta_df below and the scheduler ladder's paired_delta_table both do.
combined_results = {w: dict(native_results.get(w, {})) for w in WINDOWS_TO_RUN}
for w in WINDOWS_TO_RUN:
    combined_results[w].update(cross_results.get(w, {}))

comparison_labels = ['all']
for fam, names in FAMILIES.items():
    comparison_labels += [names['full'], names['flt'], names['flt_trg']]
    comparison_labels += [f"{names['flt']}__on_full", f"{names['flt_trg']}__on_full",
                           f"{names['full']}__on_filtered"]

metrics_df = make_metrics_table(combined_agg, WINDOWS_TO_RUN, comparison_labels, list(ALGOS), 
                                save=SAVE, save_dir=SAVE_DIR, results=combined_results)
metrics_df[['Return%', 'Sharpe', 'MaxDD%', 'Return%_mean', 'Sharpe_mean', 'MaxDD%_mean', 'Seeds']]

Return%  Sharpe  MaxDD%  Return%_mean  \
window dataset                algo                                          
w1     all                    a2c     26.23   0.935  -26.24        26.232   
                              ppo     33.21   1.755   -9.18        33.210   
                              sac     18.47   0.729  -27.11        18.472   
       mvx2                   a2c     25.52   0.918  -25.13        25.520   
                              ppo     31.42   1.824   -9.39        31.418   
...                                     ...     ...     ...           ...   
w3     flt_bollx_trg__on_full ppo     13.56   0.861  -15.24        13.554   
                              sac     14.97   0.902  -14.82        14.970   
       bollx__on_filtered     a2c     12.97   0.832  -14.52        12.972   
                              ppo     16.84   0.990  -17.19        16.844   
                              sac      9.13   0.646  -14.71         9.126   

                                    Sharpe_mean  MaxDD%_mean  Seeds  
window dataset                algo                                   
w1     all                    a2c         0.930      -26.238      5  
                              ppo         1.690       -9.938      5  
                              sac         0.716      -27.110      5  
       mvx2                   a2c         0.911      -25.134      5  
                              ppo         1.702      -10.114      5  
...                                         ...          ...    ...  
w3     flt_bollx_trg__on_full ppo         0.773      -15.284      5  
                              sac         0.793      -20.204      5  
       bollx__on_filtered     a2c         0.738      -16.884      5  
                              ppo         0.880      -18.284      5  
                              sac         0.586      -16.844      5  

[117 rows x 7 columns]

In [12]:
# Native-vs-cross delta table — the headline output
# Reads the *_mean columns (metric computed per seed, then averaged) rather than the
# plain columns (metric computed once on the seed-averaged curve) -- see the comment
# in the cell above for why that matters.
def cross_native_delta(metrics_df, fam_names, model_role, suffix, windows, algos, metric='Sharpe_mean'):
    native_ds = fam_names[model_role]
    cross_ds  = f'{native_ds}__{suffix}'
    col = metric.removesuffix('_mean')
    rows = []
    for w in windows:
        for algo in algos:
            try:
                native_val = metrics_df.loc[(w, native_ds, algo), metric]
                cross_val  = metrics_df.loc[(w, cross_ds, algo), metric]
            except KeyError:
                continue
            rows.append({'window': w, 'algo': algo, 'dataset': native_ds,
                         f'native_{col}': native_val, f'cross_{col}': cross_val,
                         'delta': cross_val - native_val})
    return pd.DataFrame(rows)

delta_rows = [
    cross_native_delta(metrics_df, names, model_role, suffix, WINDOWS_TO_RUN, list(ALGOS))
    for names in FAMILIES.values()
    for model_role, suffix in [('flt', 'on_full'), ('flt_trg', 'on_full'), ('full', 'on_filtered')]
]
delta_df = pd.concat(delta_rows, ignore_index=True)
delta_df.sort_values('delta')

,window,algo,dataset,native_Sharpe,cross_Sharpe,delta
36,w1,a2c,flt_bollx_trg,1.611,1.000,-0.611
11,w1,sac,flt_mvx2_trg,1.329,0.918,-0.411
16,w3,ppo,flt_mvx2_trg,1.480,1.081,-0.399
2,w1,sac,flt_mvx2,1.423,1.031,-0.392
27,w1,a2c,flt_bollx,1.259,0.899,-0.360
9,w1,a2c,flt_mvx2_trg,1.126,0.767,-0.359
0,w1,a2c,flt_mvx2,1.215,0.895,-0.320
38,w1,sac,flt_bollx_trg,1.557,1.258,-0.299
7,w3,ppo,flt_mvx2,1.117,0.856,-0.261
43,w3,ppo,flt_bollx_trg,0.992,0.773,-0.219


## Reading `delta_df`

`delta = cross_Sharpe - native_Sharpe`, sorted ascending (most-degraded first):

- **`flt_{fam}__on_full`** (large negative Δ): the crossover-trained agent degrades
  once evaluated on daily trading dates it never saw during training — consistent
  with, but not proof of, overfitting to the crossover schedule.
- **`flt_{fam}_trg__on_full`** (large negative Δ): the same cross-distribution drop,
  for the crossover representation specifically — indicates whether reducing the
  state to the crossover columns alone makes this worse than the indicator-and-crossover
  representation above, or no different.
- **`{fam}__on_filtered`** (large negative Δ): the daily-trained agent underperforms
  when restricted to crossover trading dates — one candidate explanation is reliance
  on the daily trading cadence that the crossover schedule does not provide, though this
  is not established here.
- A **small or positive** Δ in any row means that condition generalizes across
  schedules without a large drop.

## 2×2 ablation

### Does restricting the training dates matter once the state is already crossover-only?

Isolates the crossover representation (`{fam}_trg` / `flt_{fam}_trg`, 2 crossover columns,
state space 121) from the training-date restriction (daily vs crossover training dates).
Naming = **[TRAIN][TRADE]**, D = daily dates, E = crossover dates:

|            | trade: Daily | trade: Crossover |
|---|---|---|
| **train: Daily** (`{fam}_trg`)     | DD — native, already cached | DE — **missing, computed below** |
| **train: Crossover** (`flt_{fam}_trg`) | ED — cross, computed above  | EE — native, reconstructed above |

DE needs the same gap-aware reconstruction as the other crossover-trading-date conditions.

The table below also includes the same [TRAIN][TRADE] grid for the indicator-and-crossover
representation (`mvx2`/`bollx`, state space 181) alongside this crossover-only one —
`family` in the output has 4 values, letting you compare whether the pattern found for the
reduced state also holds once the state carries continuous indicator values.

In [13]:
def agg_metrics(seed_list):
    if not seed_list:
        return None
    if AGG_METHOD == 'mean_metrics':
        per_seed = [compute_metrics(c) for c in seed_list]
        return {k: float(np.nanmean([m[k] for m in per_seed])) for k in per_seed[0]}
    mat = pd.concat(seed_list, axis=1)
    return compute_metrics(mat.mean(axis=1))

STATE_REPRESENTATIONS = [
    ('full', 'flt'),           # hybrid state (continuous + boolean)
    ('full_trg', 'flt_trg'),   # boolean-only state
]


In [14]:
# DE/ED trade counts -- mirrors cells 12/13's cross-eval/reconstruction sweep exactly:
# same models, same shared eval dataframes (ED always evaluates on the hybrid 'full'
# role's daily df, DE always evaluates on the hybrid 'flt' role's filtered df,
# regardless of which state representation is being scored -- get_indicators() picks
# out just the columns each model needs, so the shared hybrid df is a valid source for
# both). run_cross_eval/run_eval_reconstructed discard positions there (`_, x = ...`);
# this cell captures them instead, to compute total_trades per seed. Own cache: this
# is genuinely new inference -- these cross-schedule conditions were never captured
# anywhere else. DD/EE need no new inference at all -- see native_trades_lookup()
# in the table-build cell below.
from utils.tables import compute_trade_stats

TRIGGER_TRADES_CACHE = CACHE_DIR / 'trigger2x2_trades_cache.pkl'
if not FORCE_RERUN and TRIGGER_TRADES_CACHE.exists():
    with open(TRIGGER_TRADES_CACHE, 'rb') as f:
        trigger_trades = pickle.load(f)['trades']
    print(f'✓ Loaded trigger-2x2 trades cache: {TRIGGER_TRADES_CACHE}')
else:
    trigger_trades = {}
    for w in WINDOWS_TO_RUN:
        trigger_trades[w] = {}
        for fam, names in FAMILIES.items():
            full_test_df = data[w][names['full']]['test']['df']      # shared ED eval set (hybrid daily)
            tickers = sorted(full_test_df['tic'].unique())
            full_price_wide = full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
            full_price_wide.index = pd.to_datetime(full_price_wide.index)
            full_dates = full_price_wide.index
            flt_test_df = data[w][names['flt']]['test']['df']        # shared DE eval set (hybrid filtered)

            for full_role, flt_role in STATE_REPRESENTATIONS:
                daily_ds, event_ds = names[full_role], names[flt_role]
                if daily_ds not in native_results[w]:
                    continue
                de_label = f'{daily_ds}__on_filtered'
                ed_label = f'{event_ds}__on_full'
                trigger_trades[w][de_label] = {}
                trigger_trades[w][ed_label] = {}
                for algo_name, algo_cls in ALGOS.items():
                    de_trades, ed_trades = [], []
                    for seed in SEEDS:
                        # ---- DE: daily-trained -> trigger-filtered eval (reconstructed) ----
                        de_tag = data[w][daily_ds]['train']['tag']
                        de_model = MODELS_DIR / f'{de_tag}_{algo_name}_s{seed}.zip'
                        de_vecnorm = MODELS_DIR / f'{de_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                        if de_model.exists() and de_vecnorm.exists():
                            de_indicators = get_indicators(daily_ds, df=flt_test_df)
                            _, de_positions = run_prediction_vecnorm_reconstructed(
                                de_model, de_vecnorm, algo_cls, flt_test_df, de_indicators,
                                full_price_wide, full_dates, device=DEVICE,
                            )
                            de_trades.append(compute_trade_stats(de_positions)['total_trades'])

                        # ---- ED: trigger-trained -> full daily eval (plain, no reconstruction) ----
                        ed_tag = data[w][event_ds]['train']['tag']
                        ed_model = MODELS_DIR / f'{ed_tag}_{algo_name}_s{seed}.zip'
                        ed_vecnorm = MODELS_DIR / f'{ed_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                        if ed_model.exists() and ed_vecnorm.exists():
                            ed_indicators = get_indicators(event_ds, df=full_test_df)
                            ed_env_kwargs = make_env_kwargs(full_test_df, ed_indicators)
                            df_ed = full_test_df.copy()
                            df_ed['day'] = pd.factorize(df_ed['date'])[0]
                            df_ed_indexed = df_ed.set_index('day')
                            ed_model_obj = algo_cls.load(str(ed_model), device=DEVICE)
                            ed_test_env = StockTradingEnv(df=df_ed_indexed, **ed_env_kwargs)
                            _, ed_raw = run_prediction_vecnorm(ed_model_obj, ed_test_env, ed_vecnorm)
                            ed_raw_idx = ed_raw.set_index('date') if 'date' in ed_raw.columns else ed_raw
                            ed_positions = actions_to_positions(ed_raw_idx)
                            ed_trades.append(compute_trade_stats(ed_positions)['total_trades'])
                    trigger_trades[w][de_label][algo_name] = de_trades
                    trigger_trades[w][ed_label][algo_name] = ed_trades
                print(f'  {w} {daily_ds:14s}/{event_ds:18s}: DE/ED trades done ({len(ALGOS)} algos x {len(SEEDS)} seeds)')

    with open(TRIGGER_TRADES_CACHE, 'wb') as f:
        pickle.dump({'trades': trigger_trades}, f)
    print(f'✓ Saved: {TRIGGER_TRADES_CACHE}')


✓ Loaded trigger-2x2 trades cache: /home/main/Programming/finrl/results/dow/experiment_2/trigger2x2_trades_cache.pkl


In [15]:
# Build the clean 2x2 (DD / DE / ED / EE) table, per family x state-representation x
# window x algo. Covers both state representations in one pass: the boolean-only-state
# family (mvx2_trg/bollx_trg, via the 'full_trg'/'flt_trg' roles) and the hybrid
# continuous+boolean state family (mvx2/bollx, via 'full'/'flt') -- 'family' in the
# output ends up with 4 distinct values, directly comparable side by side. All four
# cells read from native_results / cross_results only -- works identically whether
# this run just computed everything fresh or loaded a complete cache.

def native_trades_lookup(metrics_table_path=Path('tables_mv2boll/metrics_table.csv')):
    """(window, dataset, algo) -> Trades_mean/_std/_ci_lo/_ci_hi, for any dataset dow_3
    already covers natively (a model evaluated on the same kind of sequence it was
    trained on -- DD/EE here, plus every ladder stage except the two "restricted"
    ones). dow_3's own DATASETS list already covers every daily dataset and every
    flt_* dataset, 5-seed trade counts included. No new inference needed here, just
    a lookup against dow_3's saved output.
    """
    mt = pd.read_csv(metrics_table_path)
    return mt.set_index(['window', 'dataset', 'algo'])[['Trades_mean', 'Trades_std', 'Trades_ci_lo', 'Trades_ci_hi']]

# trades_lookup is kept for the ladder table further down (cell that builds
# scheduler_ladder_controls_placebo.csv reads this same variable) -- the 2x2 table's
# own Trades stats below are sourced from the raw per-seed list instead (needed
# anyway for Trades_std/_n), not from this pre-aggregated, pre-rounded CSV lookup.
trades_lookup = native_trades_lookup()

# DD/EE trades: dow_3's own 5-seed sweep already covers every daily and flt_* dataset
# -- same raw pickle cell 27 (random-2x2 table) reads, loaded again here so this
# cell's Trades stats come from the same un-rounded per-seed source as DE/ED's.
DOW3_TRADES_CACHE = FINRL_PATH / 'results' / 'dow' / 'experiment_1' / 'trade_stats_5seed_cache.pkl'
with open(DOW3_TRADES_CACHE, 'rb') as f:
    dow3_trade_stats = pickle.load(f)['trade_stats']

# Return%/Sharpe/MaxDD%/Trades each get both the existing bare column (per-seed mean,
# already correct under AGG_METHOD='mean_metrics') AND ladder-style _mean/_std/_n/
# _ci_lo/_ci_hi columns computed from the SAME raw per-seed list via tables.py's
# _seed_ci_stats/trades_ci_stats -- the exact functions the scheduler ladder table
# already uses, not a new computation path. Per-metric _n (not one blanket seed
# count) because a metric can go NaN independently of the others (e.g. Sharpe on a
# zero-volatility seed).
from utils.tables import _seed_ci_stats, trades_ci_stats

rows_2x2 = []
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        for full_role, flt_role in STATE_REPRESENTATIONS:
            daily_ds, event_ds = names[full_role], names[flt_role]
            if daily_ds not in native_results[w]:
                # experiment_1's native reference cache doesn't have every {fam} label
                # cached for every window -- skip this (family, state-representation)
                # cell rather than let one missing reference kill the whole table.
                print(f'  [skip] {w} {daily_ds}: no native reference in experiment_1 cache')
                continue
            for algo in ALGOS:
                seed_lists = {
                    'DD': native_results[w][daily_ds][algo],
                    'EE': cross_results[w][event_ds][algo],          # corrected native, saved under original label
                    'ED': cross_results[w][f'{event_ds}__on_full'][algo],
                    'DE': cross_results[w][f'{daily_ds}__on_filtered'][algo],
                }
                trade_lists = {
                    'DD': dow3_trade_stats.get(w, {}).get(daily_ds, {}).get(algo, []),
                    'EE': dow3_trade_stats.get(w, {}).get(event_ds, {}).get(algo, []),
                    'DE': trigger_trades[w].get(f'{daily_ds}__on_filtered', {}).get(algo, []),
                    'ED': trigger_trades[w].get(f'{event_ds}__on_full', {}).get(algo, []),
                }
                for cell_name, seed_list in seed_lists.items():
                    m = agg_metrics(seed_list)
                    if m is None:
                        continue
                    trade_list = trade_lists[cell_name]
                    row = {'family': daily_ds, 'window': w, 'algo': algo, 'cell': cell_name,
                           'Return%': m['Return%'], 'Sharpe': m['Sharpe'], 'MaxDD%': m['MaxDD%'],
                           'Trades': float(np.mean(trade_list)) if trade_list else np.nan}
                    row.update(_seed_ci_stats(seed_list))
                    row.update(trades_ci_stats(trade_list))
                    rows_2x2.append(row)

df_2x2 = pd.DataFrame(rows_2x2)
print(f"AGG_METHOD = '{AGG_METHOD}'  ({len(rows_2x2)} rows)")

AGG_METHOD = 'mean_metrics'  (144 rows)


In [16]:
print("=== Detailed (per window x algo) ===")

# Bare columns first (existing, per-seed mean under AGG_METHOD='mean_metrics' --
# unchanged, so downstream readers of just these 4 columns keep working), then
# ladder-style _mean/_std/_n/_ci_lo/_ci_hi for each metric, computed from the same
# raw per-seed list (see the cell above).
_stat_cols = [f'{m}{s}' for m in ['Return%', 'Sharpe', 'MaxDD%', 'Trades']
              for s in ('_mean', '_std', '_n', '_ci_lo', '_ci_hi')]
detail_cols = ['Return%', 'Sharpe', 'MaxDD%', 'Trades'] + _stat_cols

detailed = df_2x2.set_index(['family','window','algo','cell'])[detail_cols].sort_index()
print(detailed.to_string())

print("\n=== Aggregate 2x2 summary (mean across 3 windows x 3 algos) ===")
summary = df_2x2.groupby(['family','cell'])[['Return%','Sharpe','MaxDD%','Trades']].mean().round(3)
print(summary.to_string())

# AGG_METHOD is part of the filename so switching it doesn't overwrite the other
# method's saved table -- default ('mean_curve') keeps the exact filename used before
# this flag existed.
suffix = '' if AGG_METHOD == 'mean_curve' else f'_{AGG_METHOD}'
out_dir = Path('tables_mv2boll/2x2')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"2x2_train_eval_sequence{suffix}.csv"
detailed.to_csv(out_path)
print(f"\n✓ Saved detailed 2x2 table: {out_path}")

=== Detailed (per window x algo) ===
                            Return%  Sharpe  MaxDD%  Trades  Return%_mean  Return%_std  Return%_n  Return%_ci_lo  Return%_ci_hi  Sharpe_mean  Sharpe_std  Sharpe_n  Sharpe_ci_lo  Sharpe_ci_hi  MaxDD%_mean  MaxDD%_std  MaxDD%_n  MaxDD%_ci_lo  MaxDD%_ci_hi  Trades_mean  Trades_std  Trades_n  Trades_ci_lo  Trades_ci_hi
family    window algo cell                                                                                                                                                                                                                                                                                                  
bollx     w1     a2c  DD     24.780  0.9282 -27.474   115.0        24.780        9.639          5         12.811         36.749        0.928       0.360         5         0.481         1.375      -27.474       6.202         5       -35.175       -19.773        115.0      54.332         5        47.538       182.462
            

In [17]:
# 2x2 seed-level table (2x2_train_eval_sequence_seed_mean_metrics.csv) -- one row per
# (family, window, algo, cell, seed), no aggregation. DD/EE come straight from dow_3's
# metrics_seeds_table.csv (already per-seed, zero new computation); DE/ED are built
# from this notebook's own per-seed curves (cross_results) and trade counts
# (trigger_trades) -- the same raw sources the aggregated table above already uses.
metrics_seeds = pd.read_csv('tables_mv2boll/metrics_seeds_table.csv')
metrics_seeds_idx = metrics_seeds.set_index(['window', 'dataset', 'algo'])

def _cross_seed_rows(w, label, algo):
    """(seed, Return%, Sharpe, MaxDD%, Trades) rows for a DE/ED cross_results label."""
    curves = cross_results[w].get(label, {}).get(algo, [])
    trades = trigger_trades[w].get(label, {}).get(algo, [])
    if trades and len(trades) != len(curves):
        print(f'  [WARN] {w}/{label}/{algo}: {len(curves)} curves vs {len(trades)} trades -- Trades left blank')
        trades = []
    rows = []
    for i, curve in enumerate(curves):
        seed = int(curve.name[1:]) if curve.name and curve.name.startswith('s') else SEEDS[i]
        m = compute_metrics(curve)
        rows.append({'seed': seed, 'Return%': m['Return%'], 'Sharpe': m['Sharpe'],
                      'MaxDD%': m['MaxDD%'], 'Trades': trades[i] if trades else np.nan})
    return rows

seed_rows_2x2 = []
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        for full_role, flt_role in STATE_REPRESENTATIONS:
            daily_ds, event_ds = names[full_role], names[flt_role]
            if daily_ds not in native_results[w]:
                continue
            for algo in ALGOS:
                for cell_name, ds in [('DD', daily_ds), ('EE', event_ds)]:
                    if (w, ds, algo) not in metrics_seeds_idx.index:
                        continue
                    sub = metrics_seeds_idx.loc[[(w, ds, algo)]]
                    for _, r in sub.iterrows():
                        seed_rows_2x2.append({'family': daily_ds, 'window': w, 'algo': algo, 'cell': cell_name,
                                               'seed': int(r['seed']), 'Return%': r['Return%'], 'Sharpe': r['Sharpe'],
                                               'MaxDD%': r['MaxDD%'], 'Trades': r['Trades']})
                for cell_name, label in [('DE', f'{daily_ds}__on_filtered'), ('ED', f'{event_ds}__on_full')]:
                    for row in _cross_seed_rows(w, label, algo):
                        seed_rows_2x2.append({'family': daily_ds, 'window': w, 'algo': algo, 'cell': cell_name, **row})

df_2x2_seeds = pd.DataFrame(seed_rows_2x2)
seed_out_dir = Path('tables_mv2boll/2x2')
seed_out_dir.mkdir(parents=True, exist_ok=True)
seed_out_path = seed_out_dir / '2x2_train_eval_sequence_seed_mean_metrics.csv'
df_2x2_seeds.to_csv(seed_out_path, index=False)
print(f'✓ Saved: {seed_out_path}  ({len(df_2x2_seeds)} rows)')


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


... (9 outputs left out)

/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/3039785865.py:33: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, ds, algo) not in metrics_seeds_idx.index:


✓ Saved: tables_mv2boll/2x2/2x2_train_eval_sequence_seed_mean_metrics.csv  (720 rows)


## 2×2 with random-schedule groups (DD / DR / RD / RR)

Same [train][trade] structure as the crossover 2×2 above, but E (crossover) is
replaced by R (random): the date-control group's random dates instead of the
indicator-trigger dates. Reuses `dow_5_controls.ipynb`'s frozen schedules and its
separately-trained control (random-date-trained) models — no new training here, and
no re-derivation of its naming/schedule logic (`control_model_run_name`,
`manifest_row_lookup`, `load_frozen_target_dates` now live in `utils/common.py`,
shared by both notebooks). DD is already in `native_results`; RR is `dow_5_controls.ipynb`'s
own saved `seed_metrics_random_placebo.csv`, read directly below. Only DR and RD need a
new (inference-only) sweep.

In [18]:
RANDOM_MODELS_DIR = {
    'full':     FINRL_PATH / 'models' / 'dow' / 'experiment_smrsc_hybrid_v1',
    'full_trg': FINRL_PATH / 'models' / 'dow' / 'experiment_smrsc_trg_v1',
}
RANDOM_TAG_INFIX = {'full': 'placebo_hybrid', 'full_trg': 'placebo'}
DOW5_FAMILY = {'mvx2': 'moving_average', 'bollx': 'bollinger'}   # this notebook's family name -> dow_5's

# The random-date schedules are frozen once and shared across both representations'
# training runs (dow_5's own convention) -- always read from the trg_v1 results dir.
SCHEDULE_MANIFEST = FINRL_PATH / 'results' / 'dow' / 'experiment_smrsc_trg_v1' / 'schedules' / 'schedule_manifest.csv'
schedule_manifest = pd.read_csv(SCHEDULE_MANIFEST)

RANDOM_CACHE = CACHE_DIR / 'random2x2_cache.pkl'
if not FORCE_RERUN and RANDOM_CACHE.exists():
    with open(RANDOM_CACHE, 'rb') as f:
        random_results = pickle.load(f)['results']
    _random_cache_loaded = True
    print(f'✓ Loaded random-schedule cache: {RANDOM_CACHE}')
else:
    _random_cache_loaded = False
    print('[FORCE_RERUN=True] -- ignoring existing random-schedule cache.' if RANDOM_CACHE.exists()
          else 'No random-schedule cache found -- will compute DR/RD from scratch.')


✓ Loaded random-schedule cache: /home/main/Programming/finrl/results/dow/experiment_2/random2x2_cache.pkl


In [19]:
# DR: existing daily-trained model (experiment_1, no retraining) evaluated on the
# frozen random-date test panel. RD: dow_5's control (random-trained) model evaluated
# on the full daily test set. Both reconstructed the same way as the trigger 2x2's
# DE/ED cells above.
if not _random_cache_loaded:
    random_results = {}
    for w in WINDOWS_TO_RUN:
        random_results[w] = {}
        for fam, names in FAMILIES.items():
            fam5 = DOW5_FAMILY[fam]
            random_results[w][fam] = {}
            for full_role, flt_role in STATE_REPRESENTATIONS:
                daily_ds = names[full_role]
                treatment_ds = names[flt_role]
                daily_full_test_df = data[w][daily_ds]['test']['df']
                daily_tag = data[w][daily_ds]['train']['tag']
                treatment_tag = data[w][treatment_ds]['train']['tag']

                tickers = sorted(daily_full_test_df['tic'].unique())
                full_price_wide = daily_full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
                full_price_wide.index = pd.to_datetime(full_price_wide.index)
                full_dates = full_price_wide.index

                random_results[w][fam][full_role] = {}
                for algo_name, algo_cls in ALGOS.items():
                    dr_curves, rd_curves = [], []
                    for seed in SEEDS:
                        # ---- DR: daily-trained -> frozen random test panel ----
                        mrow = manifest_row_lookup(schedule_manifest, fam5, w, 'test', seed)
                        target_dates = load_frozen_target_dates('placebo', None, mrow)
                        sparse_df = daily_full_test_df[daily_full_test_df['date'].astype(str).isin(target_dates)].copy()
                        dr_indicators = get_indicators(treatment_ds, df=sparse_df)
                        dr_model = MODELS_DIR / f'{daily_tag}_{algo_name}_s{seed}.zip'
                        dr_vecnorm = MODELS_DIR / f'{daily_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                        curve, _ = run_prediction_vecnorm_reconstructed(
                            dr_model, dr_vecnorm, algo_cls, sparse_df, dr_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        dr_curves.append(curve.rename(f's{seed}'))

                        # ---- RD: control (random-trained) -> full daily test set ----
                        control_run = control_model_run_name(
                            treatment_tag, treatment_ds, RANDOM_TAG_INFIX[full_role], fam5, algo_name, seed,
                        )
                        rd_model = RANDOM_MODELS_DIR[full_role] / f'{control_run}.zip'
                        rd_vecnorm = RANDOM_MODELS_DIR[full_role] / f'{control_run}_vecnorm.pkl'
                        rd_indicators = get_indicators(treatment_ds, df=daily_full_test_df)
                        curve2, _ = run_prediction_vecnorm_reconstructed(
                            rd_model, rd_vecnorm, algo_cls, daily_full_test_df, rd_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        rd_curves.append(curve2.rename(f's{seed}'))
                    random_results[w][fam][full_role][algo_name] = {'DR': dr_curves, 'RD': rd_curves}
                print(f'  {w} {fam:14s} {full_role:9s}: DR/RD done ({len(ALGOS)} algos x {len(SEEDS)} seeds)')

    with open(RANDOM_CACHE, 'wb') as f:
        pickle.dump({'results': random_results}, f)
    print(f'✓ Saved: {RANDOM_CACHE}')
else:
    print('[cache loaded] Skipping random-schedule sweep.')


[cache loaded] Skipping random-schedule sweep.


In [20]:
# DR/RD trade counts -- same model resolution as the DR/RD sweep above, but capturing
# positions (discarded there via `curve, _ = ...`) to compute total_trades per seed.
# Own cache: genuinely new inference, positions were never kept anywhere else.
from utils.tables import compute_trade_stats

RANDOM_TRADES_CACHE = CACHE_DIR / 'random2x2_trades_cache.pkl'
if not FORCE_RERUN and RANDOM_TRADES_CACHE.exists():
    with open(RANDOM_TRADES_CACHE, 'rb') as f:
        random_trades = pickle.load(f)['trades']
    print(f'✓ Loaded random-schedule trades cache: {RANDOM_TRADES_CACHE}')
else:
    random_trades = {}
    for w in WINDOWS_TO_RUN:
        random_trades[w] = {}
        for fam, names in FAMILIES.items():
            fam5 = DOW5_FAMILY[fam]
            random_trades[w][fam] = {}
            for full_role, flt_role in STATE_REPRESENTATIONS:
                daily_ds = names[full_role]
                treatment_ds = names[flt_role]
                daily_full_test_df = data[w][daily_ds]['test']['df']
                daily_tag = data[w][daily_ds]['train']['tag']
                treatment_tag = data[w][treatment_ds]['train']['tag']

                tickers = sorted(daily_full_test_df['tic'].unique())
                full_price_wide = daily_full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
                full_price_wide.index = pd.to_datetime(full_price_wide.index)
                full_dates = full_price_wide.index

                random_trades[w][fam][full_role] = {}
                for algo_name, algo_cls in ALGOS.items():
                    dr_trades, rd_trades = [], []
                    for seed in SEEDS:
                        # ---- DR trades ----
                        mrow = manifest_row_lookup(schedule_manifest, fam5, w, 'test', seed)
                        target_dates = load_frozen_target_dates('placebo', None, mrow)
                        sparse_df = daily_full_test_df[daily_full_test_df['date'].astype(str).isin(target_dates)].copy()
                        dr_indicators = get_indicators(treatment_ds, df=sparse_df)
                        dr_model = MODELS_DIR / f'{daily_tag}_{algo_name}_s{seed}.zip'
                        dr_vecnorm = MODELS_DIR / f'{daily_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                        _, dr_positions = run_prediction_vecnorm_reconstructed(
                            dr_model, dr_vecnorm, algo_cls, sparse_df, dr_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        dr_trades.append(compute_trade_stats(dr_positions)['total_trades'])

                        # ---- RD trades ----
                        control_run = control_model_run_name(
                            treatment_tag, treatment_ds, RANDOM_TAG_INFIX[full_role], fam5, algo_name, seed,
                        )
                        rd_model = RANDOM_MODELS_DIR[full_role] / f'{control_run}.zip'
                        rd_vecnorm = RANDOM_MODELS_DIR[full_role] / f'{control_run}_vecnorm.pkl'
                        rd_indicators = get_indicators(treatment_ds, df=daily_full_test_df)
                        _, rd_positions = run_prediction_vecnorm_reconstructed(
                            rd_model, rd_vecnorm, algo_cls, daily_full_test_df, rd_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        rd_trades.append(compute_trade_stats(rd_positions)['total_trades'])
                    random_trades[w][fam][full_role][algo_name] = {'DR': dr_trades, 'RD': rd_trades}
                print(f'  {w} {fam:14s} {full_role:9s}: DR/RD trades done ({len(ALGOS)} algos x {len(SEEDS)} seeds)')

    with open(RANDOM_TRADES_CACHE, 'wb') as f:
        pickle.dump({'trades': random_trades}, f)
    print(f'✓ Saved: {RANDOM_TRADES_CACHE}')


✓ Loaded random-schedule trades cache: /home/main/Programming/finrl/results/dow/experiment_2/random2x2_trades_cache.pkl


In [21]:
# DD/DR/RD/RR table -- DD from native_results (already loaded), RR read directly from
# dow_5's own saved placebo tables (no recomputation), DR/RD from random_results above.
# agg_metrics (defined above, AGG_METHOD-aware) is reused for DD/DR/RD; RR's numbers
# come pre-computed per seed in dow_5's CSV (mean_metrics-equivalent by construction),
# so they're averaged directly regardless of AGG_METHOD.
#
# Trades: DD's come from dow_3_backtest.ipynb's own 5-seed trade-stats cache (mvx2/
# bollx/mvx2_trg/bollx_trg are all in dow_3's DATASETS, no new inference needed here);
# RR's from dow_5's seed_metrics CSV (already has a 'trades' column, computed there);
# DR/RD's from the new trades-sweep cell above (genuinely new -- these cross-schedule
# conditions exist nowhere else).
RANDOM_PLACEBO_CSV = {
    'full':     Path('tables_mv2boll/experiment_smrsc_hybrid_runs/v1/seed_metrics_random_placebo.csv'),
    'full_trg': Path('tables_mv2boll/experiment_smrsc_trg_runs/v1/seed_metrics_random_placebo.csv'),
}
rr_dfs = {role: pd.read_csv(path) for role, path in RANDOM_PLACEBO_CSV.items()}

DOW3_TRADES_CACHE = FINRL_PATH / 'results' / 'dow' / 'experiment_1' / 'trade_stats_5seed_cache.pkl'
with open(DOW3_TRADES_CACHE, 'rb') as f:
    dow3_trade_stats = pickle.load(f)['trade_stats']

# RR's per-seed values come pre-computed in dow_5's CSV (one row per seed already --
# never an equity curve here), so _seed_ci_stats (which expects curves and runs
# compute_metrics on each) can't be reused directly for Return%/Sharpe/MaxDD% --
# this small inline function mirrors its exact mean/std/ci/n formula (same _t_crit)
# against a plain values array instead. Trades has no such problem: trades_ci_stats
# already takes a plain list, so RR's 'trades' column plugs straight into it.
from utils.tables import _seed_ci_stats, trades_ci_stats, _t_crit

def _rr_ci_stats(rr_seeds):
    col_map = {'Return%': 'return', 'Sharpe': 'daily_sharpe', 'MaxDD%': 'max_drawdown'}
    out = {}
    for m, col in col_map.items():
        vals = rr_seeds[col].dropna().to_numpy(dtype=float)
        n = len(vals)
        out[f'{m}_n'] = int(n)
        if n == 0:
            out[f'{m}_mean'] = out[f'{m}_std'] = out[f'{m}_ci_lo'] = out[f'{m}_ci_hi'] = np.nan
        elif n == 1:
            out[f'{m}_mean'] = round(float(vals[0]), 3)
            out[f'{m}_std'] = out[f'{m}_ci_lo'] = out[f'{m}_ci_hi'] = np.nan
        else:
            mean, std = vals.mean(), vals.std(ddof=1)
            se, t = std / np.sqrt(n), _t_crit(n)
            out[f'{m}_mean']  = round(float(mean), 3)
            out[f'{m}_std']   = round(float(std), 3)
            out[f'{m}_ci_lo'] = round(float(mean - t * se), 3)
            out[f'{m}_ci_hi'] = round(float(mean + t * se), 3)
    return out

rows_random2x2 = []
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        fam5 = DOW5_FAMILY[fam]
        for full_role, flt_role in STATE_REPRESENTATIONS:
            daily_ds = names[full_role]
            for algo in ALGOS:
                seed_lists = {
                    'DD': native_results[w][daily_ds][algo],
                    'DR': random_results[w][fam][full_role][algo]['DR'],
                    'RD': random_results[w][fam][full_role][algo]['RD'],
                }
                rr_sub = rr_dfs[full_role]
                rr_seeds = rr_sub[(rr_sub['family'] == fam5) & (rr_sub['window'] == w) & (rr_sub['algorithm'] == algo)
                                   & (rr_sub['condition'] == 'random_date_trigger_panel_placebo')]
                RR = ({'Return%': rr_seeds['return'].mean(), 'Sharpe': rr_seeds['daily_sharpe'].mean(),
                       'MaxDD%': rr_seeds['max_drawdown'].mean()} if len(rr_seeds) else None)

                dd_trades_list = dow3_trade_stats.get(w, {}).get(daily_ds, {}).get(algo, [])
                rr_trades_list = rr_seeds['trades'].dropna().tolist() if len(rr_seeds) else []
                trade_lists = {
                    'DD': dd_trades_list,
                    'DR': random_trades[w][fam][full_role][algo]['DR'],
                    'RD': random_trades[w][fam][full_role][algo]['RD'],
                    'RR': rr_trades_list,
                }

                for cell_name, m in [('DD', agg_metrics(seed_lists['DD'])), ('DR', agg_metrics(seed_lists['DR'])),
                                      ('RD', agg_metrics(seed_lists['RD'])), ('RR', RR)]:
                    if m is None:
                        continue
                    trade_list = trade_lists[cell_name]
                    row = {'family': daily_ds, 'window': w, 'algo': algo, 'cell': cell_name,
                           'Return%': m['Return%'], 'Sharpe': m['Sharpe'], 'MaxDD%': m['MaxDD%'],
                           'Trades': float(np.mean(trade_list)) if len(trade_list) else np.nan}
                    row.update(_rr_ci_stats(rr_seeds) if cell_name == 'RR' else _seed_ci_stats(seed_lists[cell_name]))
                    row.update(trades_ci_stats(trade_list))
                    rows_random2x2.append(row)

df_random2x2 = pd.DataFrame(rows_random2x2)
print(f"AGG_METHOD = '{AGG_METHOD}'  ({len(rows_random2x2)} rows)")

_stat_cols = [f'{m}{s}' for m in ['Return%', 'Sharpe', 'MaxDD%', 'Trades']
              for s in ('_mean', '_std', '_n', '_ci_lo', '_ci_hi')]
detail_cols_random = ['Return%', 'Sharpe', 'MaxDD%', 'Trades'] + _stat_cols

print("\n=== Detailed (per window x algo) ===")
detailed_random = df_random2x2.set_index(['family', 'window', 'algo', 'cell'])[detail_cols_random].sort_index()
print(detailed_random.to_string())

print("\n=== Aggregate summary (mean across 3 windows x 3 algos) ===")
summary_random = df_random2x2.groupby(['family', 'cell'])[['Return%', 'Sharpe', 'MaxDD%', 'Trades']].mean().round(3)
print(summary_random.to_string())

suffix = '' if AGG_METHOD == 'mean_curve' else f'_{AGG_METHOD}'
random2x2_out_dir = Path('tables_mv2boll/2x2')
random2x2_out_dir.mkdir(parents=True, exist_ok=True)
out_path = random2x2_out_dir / f"random2x2_train_eval_sequence{suffix}.csv"
detailed_random.to_csv(out_path)
print(f"\n✓ Saved: {out_path}")

# Paired significance -- same training-side/eval-side split as the trigger 2x2:
# RD-DD/RR-DR hold eval fixed (does TRAIN distribution matter), DR-DD/RR-RD hold
# train fixed (does switching EVAL to the random schedule matter). n=18
# (2 families x 3 windows x 3 algos) per comparison, per state representation.
from scipy import stats as _stats

def _paired_ci(diffs):
    diffs = np.array(diffs); n = len(diffs)
    mean = diffs.mean(); se = diffs.std(ddof=1) / np.sqrt(n)
    t = _stats.t.ppf(0.975, n - 1)
    return mean, mean - t * se, mean + t * se, int((diffs > 0).sum()), n

STATE_FAMS = {'hybrid': [names['full'] for names in FAMILIES.values()],
              'binary': [names['full_trg'] for names in FAMILIES.values()]}
comparisons = {
    'training-side: RD - DD (eval=Daily fixed)':  ('RD', 'DD'),
    'training-side: RR - DR (eval=Random fixed)': ('RR', 'DR'),
    'eval-side: DR - DD (train=Daily fixed)':      ('DR', 'DD'),
    'eval-side: RR - RD (train=Random fixed)':     ('RR', 'RD'),
}
sig_rows = []
for state, fams in STATE_FAMS.items():
    sub = df_random2x2[df_random2x2['family'].isin(fams)]
    piv = sub.set_index(['family', 'window', 'algo', 'cell'])[['Return%', 'Sharpe', 'MaxDD%', 'Trades']].unstack('cell')
    for comp_name, (a, b) in comparisons.items():
        for m in ['Sharpe', 'Return%', 'MaxDD%', 'Trades']:
            d = (piv[(m, a)] - piv[(m, b)]).dropna()
            mean, lo, hi, wins, n = _paired_ci(d.values)
            sig_rows.append({'state': state, 'comparison': comp_name, 'metric': m,
                              'mean_diff': round(mean, 4), 'ci_lo': round(lo, 4), 'ci_hi': round(hi, 4),
                              'wins': wins, 'n': n, 'reliable': (lo > 0) or (hi < 0)})

df_random2x2_sig = pd.DataFrame(sig_rows)
sig_out_path = f"tables_mv2boll/random2x2_significance{suffix}.csv"
df_random2x2_sig.to_csv(sig_out_path, index=False)
print("\n=== Significance (paired, n=18, 95% CI) ===")
print(df_random2x2_sig.to_string(index=False))
print(f"\n✓ Saved: {sig_out_path}")


AGG_METHOD = 'mean_metrics'  (144 rows)

=== Detailed (per window x algo) ===
                            Return%  Sharpe  MaxDD%  Trades  Return%_mean  Return%_std  Return%_n  Return%_ci_lo  Return%_ci_hi  Sharpe_mean  Sharpe_std  Sharpe_n  Sharpe_ci_lo  Sharpe_ci_hi  MaxDD%_mean  MaxDD%_std  MaxDD%_n  MaxDD%_ci_lo  MaxDD%_ci_hi  Trades_mean  Trades_std  Trades_n  Trades_ci_lo  Trades_ci_hi
family    window algo cell                                                                                                                                                                                                                                                                                                  
bollx     w1     a2c  DD     24.780  0.9282 -27.474   115.0        24.780        9.639          5         12.811         36.749        0.928       0.360         5         0.481         1.375      -27.474       6.202         5       -35.175       -19.773        115.0      54.332         5

In [22]:
# random-2x2 seed-level table (random2x2_train_eval_sequence_seed_mean_metrics.csv)
# -- one row per (family, window, algo, cell, seed). DD reuses metrics_seeds_idx
# (built in the 2x2 seed-level cell above); DR/RD are built from this notebook's own
# per-seed curves (random_results) and trade counts (random_trades); RR is copied
# straight from dow_5's own per-seed CSV (rr_dfs, already exactly this shape), just
# with columns renamed to match.
def _curve_trade_seed_rows(curves, trades, warn_label):
    if trades and len(trades) != len(curves):
        print(f'  [WARN] {warn_label}: {len(curves)} curves vs {len(trades)} trades -- Trades left blank')
        trades = []
    rows = []
    for i, curve in enumerate(curves):
        seed = int(curve.name[1:]) if curve.name and curve.name.startswith('s') else SEEDS[i]
        m = compute_metrics(curve)
        rows.append({'seed': seed, 'Return%': m['Return%'], 'Sharpe': m['Sharpe'],
                      'MaxDD%': m['MaxDD%'], 'Trades': trades[i] if trades else np.nan})
    return rows

seed_rows_random2x2 = []
for w in WINDOWS_TO_RUN:
    for fam, names in FAMILIES.items():
        fam5 = DOW5_FAMILY[fam]
        for full_role, flt_role in STATE_REPRESENTATIONS:
            daily_ds = names[full_role]
            for algo in ALGOS:
                if (w, daily_ds, algo) in metrics_seeds_idx.index:
                    sub = metrics_seeds_idx.loc[[(w, daily_ds, algo)]]
                    for _, r in sub.iterrows():
                        seed_rows_random2x2.append({'family': daily_ds, 'window': w, 'algo': algo, 'cell': 'DD',
                                                     'seed': int(r['seed']), 'Return%': r['Return%'], 'Sharpe': r['Sharpe'],
                                                     'MaxDD%': r['MaxDD%'], 'Trades': r['Trades']})
                for cell_name in ['DR', 'RD']:
                    curves = random_results[w][fam][full_role][algo][cell_name]
                    trades = random_trades[w][fam][full_role][algo][cell_name]
                    warn_label = f'{w}/{daily_ds}/{algo}/{cell_name}'
                    for row in _curve_trade_seed_rows(curves, trades, warn_label):
                        seed_rows_random2x2.append({'family': daily_ds, 'window': w, 'algo': algo, 'cell': cell_name, **row})

                rr_sub = rr_dfs[full_role]
                rr_seeds = rr_sub[(rr_sub['family'] == fam5) & (rr_sub['window'] == w) & (rr_sub['algorithm'] == algo)
                                   & (rr_sub['condition'] == 'random_date_trigger_panel_placebo')]
                for _, r in rr_seeds.iterrows():
                    seed_rows_random2x2.append({'family': daily_ds, 'window': w, 'algo': algo, 'cell': 'RR',
                                                 'seed': int(r['seed']), 'Return%': r['return'], 'Sharpe': r['daily_sharpe'],
                                                 'MaxDD%': r['max_drawdown'], 'Trades': r['trades']})

df_random2x2_seeds = pd.DataFrame(seed_rows_random2x2)
seed_out_path = seed_out_dir / 'random2x2_train_eval_sequence_seed_mean_metrics.csv'
df_random2x2_seeds.to_csv(seed_out_path, index=False)
print(f'✓ Saved: {seed_out_path}  ({len(df_random2x2_seeds)} rows)')


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


... (9 outputs left out)

/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:
/tmp/ipykernel_35875/2829182520.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, daily_ds, algo) in metrics_seeds_idx.index:


✓ Saved: tables_mv2boll/2x2/random2x2_train_eval_sequence_seed_mean_metrics.csv  (720 rows)


## Trigger-scheduler diagnostic ladder

For each family, the comparison keeps the fixed rule unchanged and moves from a daily state without trigger flags to the full scheduler treatment. The key control is a daily-trained agent with no trigger flags that is evaluated only on selected dates. This isolates the decision-date restriction. All filtered evaluations are reconstructed on the complete daily calendar.


In [23]:
# Train neither new models nor alter saved models. Evaluate daily-trained baseline
# agents on the same selected dates used by their matching trigger schedulers.
import gc
SCHEDULER_FAMILIES = {
    'moving_average': {
        'baseline': 'mv2', 'daily_hybrid': 'mvx2',
        'filtered_hybrid': 'flt_mvx2', 'filtered_binary': 'flt_mvx2_trg',
    },
    'bollinger': {
        'baseline': 'boll', 'daily_hybrid': 'bollx',
        'filtered_hybrid': 'flt_bollx', 'filtered_binary': 'flt_bollx_trg',
    },
}
# Also restrict the no-trigger-flag baseline to a random size-matched panel (the same
# frozen schedule the random-2x2 section above reads), alongside the existing
# trigger-panel restriction -- isolates decision-date RESTRICTION from decision-date
# CHOICE, with a state that has zero trigger awareness either way. Set False to skip
# (cheaper: one fewer condition per family/window/algo/seed).
INCLUDE_RANDOM_PLACEBO_STAGE = True

baseline_data = make_window_data(
    [spec['baseline'] for spec in SCHEDULER_FAMILIES.values()],
    windows=WINDOWS_TO_RUN, cvtd=False, splits=['train'],
)
for w in WINDOWS_TO_RUN:
    data[w].update(baseline_data[w])

ladder_control_results = {w: {} for w in WINDOWS_TO_RUN}
for family, spec in SCHEDULER_FAMILIES.items():
    baseline = spec['baseline']
    selected = spec['filtered_hybrid']
    restricted_label = f'{baseline}__on_{selected}'
    random_label = f'{baseline}__on_random'

    for w in WINDOWS_TO_RUN:
        # The daily-hybrid dataframe supplies the complete daily price calendar.
        full_test_df = data[w][spec['daily_hybrid']]['test']['df']
        tickers = sorted(full_test_df['tic'].unique())
        full_price_wide = full_test_df.pivot(
            index='date', columns='tic', values='close'
        )[tickers]
        full_price_wide.index = pd.to_datetime(full_price_wide.index)
        full_dates = full_price_wide.index
        selected_eval_df = data[w][selected]['test']['df']

        ladder_control_results[w][baseline] = {}
        ladder_control_results[w][restricted_label] = {}
        if INCLUDE_RANDOM_PLACEBO_STAGE:
            ladder_control_results[w][random_label] = {}
        for algo_name, algo_cls in ALGOS.items():
            daily_curves, restricted_curves, random_curves = [], [], []
            for seed in SEEDS:
                # Daily baseline: no trigger flags and a decision on every date.
                daily_curve = run_cross_eval(
                    baseline, full_test_df, w, algo_name, algo_cls, seed,
                )
                if daily_curve is not None:
                    daily_curves.append(daily_curve)

                # Same model and inputs; only decision dates are restricted.
                restricted_curve = run_eval_reconstructed(
                    baseline, selected_eval_df, w, algo_name, algo_cls, seed,
                    full_price_wide, full_dates,
                )
                if restricted_curve is not None:
                    restricted_curves.append(restricted_curve)

                if INCLUDE_RANDOM_PLACEBO_STAGE:
                    # Same model, restricted instead to the frozen RANDOM panel --
                    # isolates decision-date restriction from which dates get picked.
                    mrow = manifest_row_lookup(schedule_manifest, family, w, 'test', seed)
                    random_dates = load_frozen_target_dates('placebo', None, mrow)
                    random_eval_df = full_test_df[full_test_df['date'].astype(str).isin(random_dates)]
                    random_curve = run_eval_reconstructed(
                        baseline, random_eval_df, w, algo_name, algo_cls, seed,
                        full_price_wide, full_dates,
                    )
                    if random_curve is not None:
                        random_curves.append(random_curve)
                    del random_curve

                # FinRL environments contain cyclic references; collect them before
                # loading the next seed model in this long inference-only sweep.
                del daily_curve, restricted_curve
                gc.collect()

            ladder_control_results[w][baseline][algo_name] = daily_curves
            ladder_control_results[w][restricted_label][algo_name] = restricted_curves
            if INCLUDE_RANDOM_PLACEBO_STAGE:
                ladder_control_results[w][random_label][algo_name] = random_curves
            print(
                f'{family:16s} {w} {algo_name}: daily {len(daily_curves)}/{len(SEEDS)}; '
                f'restricted {len(restricted_curves)}/{len(SEEDS)}'
                + (f'; random {len(random_curves)}/{len(SEEDS)}' if INCLUDE_RANDOM_PLACEBO_STAGE else '')
            )


Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121
Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121
Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121
Stock Dimension: 30, State Space: 121


... (203 outputs left out)

Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121


Stock Dimension: 30, State Space: 121


bollinger        w3 sac: daily 5/5; restricted 5/5; random 5/5


In [24]:
# Trades for the two "restricted" ladder stages (baseline model, forced onto either
# the trigger schedule or the frozen random schedule) -- mirrors the cell above's
# restricted/random curve computation exactly (same baseline model, same eval
# dataframes), but calls run_prediction_vecnorm_reconstructed directly to capture
# positions (run_eval_reconstructed discards them there) and compute total_trades per
# seed. The other four ladder stages (daily baseline, daily hybrid, scheduler hybrid,
# scheduler binary state) need no new inference at all -- their datasets are already
# in dow_3's DATASETS list, trades and all -- see native_trades_lookup() above.
LADDER_TRADES_CACHE = CACHE_DIR / 'ladder_trades_cache.pkl'
if not FORCE_RERUN and LADDER_TRADES_CACHE.exists():
    with open(LADDER_TRADES_CACHE, 'rb') as f:
        ladder_trades = pickle.load(f)['trades']
    print(f'✓ Loaded ladder trades cache: {LADDER_TRADES_CACHE}')
else:
    ladder_trades = {w: {} for w in WINDOWS_TO_RUN}
    for family, spec in SCHEDULER_FAMILIES.items():
        baseline = spec['baseline']
        selected = spec['filtered_hybrid']
        restricted_label = f'{baseline}__on_{selected}'
        random_label = f'{baseline}__on_random'

        for w in WINDOWS_TO_RUN:
            full_test_df = data[w][spec['daily_hybrid']]['test']['df']
            tickers = sorted(full_test_df['tic'].unique())
            full_price_wide = full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
            full_price_wide.index = pd.to_datetime(full_price_wide.index)
            full_dates = full_price_wide.index
            selected_eval_df = data[w][selected]['test']['df']
            train_tag = data[w][baseline]['train']['tag']

            ladder_trades[w][restricted_label] = {}
            if INCLUDE_RANDOM_PLACEBO_STAGE:
                ladder_trades[w][random_label] = {}
            for algo_name, algo_cls in ALGOS.items():
                restricted_trades, random_trades = [], []
                for seed in SEEDS:
                    model_path = MODELS_DIR / f'{train_tag}_{algo_name}_s{seed}.zip'
                    vecnorm_path = MODELS_DIR / f'{train_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                    if not (model_path.exists() and vecnorm_path.exists()):
                        continue

                    restricted_indicators = get_indicators(baseline, df=selected_eval_df)
                    _, restricted_positions = run_prediction_vecnorm_reconstructed(
                        model_path, vecnorm_path, algo_cls, selected_eval_df, restricted_indicators,
                        full_price_wide, full_dates, device=DEVICE,
                    )
                    restricted_trades.append(compute_trade_stats(restricted_positions)['total_trades'])

                    if INCLUDE_RANDOM_PLACEBO_STAGE:
                        mrow = manifest_row_lookup(schedule_manifest, family, w, 'test', seed)
                        random_dates = load_frozen_target_dates('placebo', None, mrow)
                        random_eval_df = full_test_df[full_test_df['date'].astype(str).isin(random_dates)]
                        random_indicators = get_indicators(baseline, df=random_eval_df)
                        _, random_positions = run_prediction_vecnorm_reconstructed(
                            model_path, vecnorm_path, algo_cls, random_eval_df, random_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        random_trades.append(compute_trade_stats(random_positions)['total_trades'])
                ladder_trades[w][restricted_label][algo_name] = restricted_trades
                if INCLUDE_RANDOM_PLACEBO_STAGE:
                    ladder_trades[w][random_label][algo_name] = random_trades
            print(
                f'  {family:16s} {w}: restricted trades done'
                + (' + random trades done' if INCLUDE_RANDOM_PLACEBO_STAGE else '')
                + f' ({len(ALGOS)} algos x {len(SEEDS)} seeds)'
            )

    with open(LADDER_TRADES_CACHE, 'wb') as f:
        pickle.dump({'trades': ladder_trades}, f)
    print(f'✓ Saved: {LADDER_TRADES_CACHE}')


✓ Loaded ladder trades cache: /home/main/Programming/finrl/results/dow/experiment_2/ladder_trades_cache.pkl


## 'all' (full indicator set) vs crossover-restricted vs random-restricted

Same comparison as the ladder above (daily trading dates → crossover trading dates →
random trading dates), but for the 'all' dataset (`INDICATORS_ALL`:
macd/boll_ub/boll_lb/rsi_30/cci_30/dx_30/close_30_sma/close_60_sma), daily-trained, with
no crossover columns at all. 'all' isn't tied to one crossover family, so it is evaluated
against both families' trading-date schedules (same model, two sets of trading dates each),
reusing the existing 'all'-trained models from `experiment_1` and the same frozen random
schedule as everywhere else. No new training.

In [25]:
INCLUDE_ALL_INDICATORS_STAGE = True

if INCLUDE_ALL_INDICATORS_STAGE:
    all_indicators_results = {w: {} for w in WINDOWS_TO_RUN}
    for family, spec in SCHEDULER_FAMILIES.items():
        selected = spec['filtered_hybrid']
        trigger_label = f'all__on_{selected}'
        random_label = f'all__on_random_{family}'

        for w in WINDOWS_TO_RUN:
            full_test_df = data[w][spec['daily_hybrid']]['test']['df']
            tickers = sorted(full_test_df['tic'].unique())
            full_price_wide = full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
            full_price_wide.index = pd.to_datetime(full_price_wide.index)
            full_dates = full_price_wide.index
            # 'all' uses INDICATORS_ALL (macd/rsi_30/cci_30/...), a completely different
            # engineered feature set from flt_mvx2/flt_bollx's own moving-average/Bollinger
            # columns -- can't reuse their dataframe directly (wrong columns for this
            # model). Take just their DATES, then filter 'all's own full test df to those.
            all_full_test_df = data[w]['all']['test']['df']
            trigger_dates = set(data[w][selected]['test']['df']['date'].astype(str))
            trigger_eval_df = all_full_test_df[all_full_test_df['date'].astype(str).isin(trigger_dates)]

            all_indicators_results[w][trigger_label] = {}
            all_indicators_results[w][random_label] = {}
            for algo_name, algo_cls in ALGOS.items():
                trigger_curves, random_curves = [], []
                for seed in SEEDS:
                    trigger_curve = run_eval_reconstructed(
                        'all', trigger_eval_df, w, algo_name, algo_cls, seed,
                        full_price_wide, full_dates,
                    )
                    if trigger_curve is not None:
                        trigger_curves.append(trigger_curve)

                    mrow = manifest_row_lookup(schedule_manifest, family, w, 'test', seed)
                    random_dates = load_frozen_target_dates('placebo', None, mrow)
                    random_eval_df = all_full_test_df[all_full_test_df['date'].astype(str).isin(random_dates)]
                    random_curve = run_eval_reconstructed(
                        'all', random_eval_df, w, algo_name, algo_cls, seed,
                        full_price_wide, full_dates,
                    )
                    if random_curve is not None:
                        random_curves.append(random_curve)

                all_indicators_results[w][trigger_label][algo_name] = trigger_curves
                all_indicators_results[w][random_label][algo_name] = random_curves
            print(f'  {family:16s} {w}: all-on-trigger and all-on-random done ({len(ALGOS)} algos x {len(SEEDS)} seeds)')
else:
    print('INCLUDE_ALL_INDICATORS_STAGE = False -- skipping.')


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301
Stock Dimension: 30, State Space: 301


... (115 outputs left out)

Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301


Stock Dimension: 30, State Space: 301


  bollinger        w3: all-on-trigger and all-on-random done (3 algos x 5 seeds)


In [26]:
# Trades for the 'all' (full indicator set) benchmark's two restricted stages --
# mirrors the cell above's all-on-trigger/all-on-random curve computation exactly
# (same 'all'-trained model, same eval dataframes), but calls
# run_prediction_vecnorm_reconstructed directly to capture positions
# (run_eval_reconstructed discards them there). Folds straight into ladder_trades
# (built by the ladder-trades-sweep cell earlier) so the five-stage table cell below
# picks these up with its existing Trades-merge logic -- no changes needed there.
if INCLUDE_ALL_INDICATORS_STAGE:
    ALL_INDICATORS_TRADES_CACHE = CACHE_DIR / 'all_indicators_trades_cache.pkl'
    if not FORCE_RERUN and ALL_INDICATORS_TRADES_CACHE.exists():
        with open(ALL_INDICATORS_TRADES_CACHE, 'rb') as f:
            all_indicators_trades = pickle.load(f)['trades']
        print(f'✓ Loaded all-indicators trades cache: {ALL_INDICATORS_TRADES_CACHE}')
    else:
        all_indicators_trades = {w: {} for w in WINDOWS_TO_RUN}
        for family, spec in SCHEDULER_FAMILIES.items():
            selected = spec['filtered_hybrid']
            trigger_label = f'all__on_{selected}'
            random_label = f'all__on_random_{family}'

            for w in WINDOWS_TO_RUN:
                full_test_df = data[w][spec['daily_hybrid']]['test']['df']
                tickers = sorted(full_test_df['tic'].unique())
                full_price_wide = full_test_df.pivot(index='date', columns='tic', values='close')[tickers]
                full_price_wide.index = pd.to_datetime(full_price_wide.index)
                full_dates = full_price_wide.index
                all_full_test_df = data[w]['all']['test']['df']
                trigger_dates = set(data[w][selected]['test']['df']['date'].astype(str))
                trigger_eval_df = all_full_test_df[all_full_test_df['date'].astype(str).isin(trigger_dates)]
                train_tag = data[w]['all']['train']['tag']

                all_indicators_trades[w][trigger_label] = {}
                all_indicators_trades[w][random_label] = {}
                for algo_name, algo_cls in ALGOS.items():
                    trigger_trades, random_trades = [], []
                    for seed in SEEDS:
                        model_path = MODELS_DIR / f'{train_tag}_{algo_name}_s{seed}.zip'
                        vecnorm_path = MODELS_DIR / f'{train_tag}_{algo_name}_s{seed}_vecnorm.pkl'
                        if not (model_path.exists() and vecnorm_path.exists()):
                            continue

                        trigger_indicators = get_indicators('all', df=trigger_eval_df)
                        _, trigger_positions = run_prediction_vecnorm_reconstructed(
                            model_path, vecnorm_path, algo_cls, trigger_eval_df, trigger_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        trigger_trades.append(compute_trade_stats(trigger_positions)['total_trades'])

                        mrow = manifest_row_lookup(schedule_manifest, family, w, 'test', seed)
                        random_dates = load_frozen_target_dates('placebo', None, mrow)
                        random_eval_df = all_full_test_df[all_full_test_df['date'].astype(str).isin(random_dates)]
                        random_indicators = get_indicators('all', df=random_eval_df)
                        _, random_positions = run_prediction_vecnorm_reconstructed(
                            model_path, vecnorm_path, algo_cls, random_eval_df, random_indicators,
                            full_price_wide, full_dates, device=DEVICE,
                        )
                        random_trades.append(compute_trade_stats(random_positions)['total_trades'])
                    all_indicators_trades[w][trigger_label][algo_name] = trigger_trades
                    all_indicators_trades[w][random_label][algo_name] = random_trades
                print(f'  {family:16s} {w}: all-on-trigger/all-on-random trades done ({len(ALGOS)} algos x {len(SEEDS)} seeds)')

        with open(ALL_INDICATORS_TRADES_CACHE, 'wb') as f:
            pickle.dump({'trades': all_indicators_trades}, f)
        print(f'✓ Saved: {ALL_INDICATORS_TRADES_CACHE}')

    for w in WINDOWS_TO_RUN:
        ladder_trades[w].update(all_indicators_trades[w])
    print('✓ Folded all-indicators trades into ladder_trades.')


✓ Loaded all-indicators trades cache: /home/main/Programming/finrl/results/dow/experiment_2/all_indicators_trades_cache.pkl
✓ Folded all-indicators trades into ladder_trades.


In [27]:
# Five-stage table per family (six if INCLUDE_RANDOM_PLACEBO_STAGE). The preceding
# cells must have run so that combined_agg and cross_results contain reconstructed
# scheduler curves.
from utils.tables import make_metrics_table

ladder_agg = aggregate_seeds(ladder_control_results)
for w in WINDOWS_TO_RUN:
    combined_agg[w].update(ladder_agg[w])

if INCLUDE_ALL_INDICATORS_STAGE:
    all_indicators_agg = aggregate_seeds(all_indicators_results)
    for w in WINDOWS_TO_RUN:
        combined_agg[w].update(all_indicators_agg[w])

stage_order = ['daily baseline', 'daily hybrid', 'baseline restricted to selected dates']
if INCLUDE_RANDOM_PLACEBO_STAGE:
    stage_order.append('baseline restricted to random dates')
stage_order += ['scheduler hybrid', 'scheduler binary state']
if INCLUDE_ALL_INDICATORS_STAGE:
    stage_order += ['daily benchmark', 'daily benchmark restricted to selected dates',
                     'daily benchmark restricted to random dates']

stage_rows = []
for family, spec in SCHEDULER_FAMILIES.items():
    baseline = spec['baseline']
    selected = spec['filtered_hybrid']
    stage_rows.extend([
        {'family': family, 'stage': 'daily baseline', 'dataset': baseline},
        {'family': family, 'stage': 'daily hybrid', 'dataset': spec['daily_hybrid']},
        {'family': family, 'stage': 'baseline restricted to selected dates',
         'dataset': f'{baseline}__on_{selected}'},
    ])
    if INCLUDE_RANDOM_PLACEBO_STAGE:
        stage_rows.append(
            {'family': family, 'stage': 'baseline restricted to random dates',
             'dataset': f'{baseline}__on_random'},
        )
    stage_rows.extend([
        {'family': family, 'stage': 'scheduler hybrid', 'dataset': selected},
        {'family': family, 'stage': 'scheduler binary state',
         'dataset': spec['filtered_binary']},
    ])

if INCLUDE_ALL_INDICATORS_STAGE:
    # 'all' isn't tied to one family (same model for both) -- its own pseudo-family
    # keeps it a single row per window/algo instead of duplicating it under both.
    stage_rows.append({'family': 'all', 'stage': 'daily benchmark', 'dataset': 'all'})
    for family, spec in SCHEDULER_FAMILIES.items():
        selected = spec['filtered_hybrid']
        stage_rows.extend([
            {'family': family, 'stage': 'daily benchmark restricted to selected dates',
             'dataset': f'all__on_{selected}'},
            {'family': family, 'stage': 'daily benchmark restricted to random dates',
             'dataset': f'all__on_random_{family}'},
        ])

ladder_labels = [row['dataset'] for row in stage_rows]
ladder_seed_results = {w: {} for w in WINDOWS_TO_RUN}
for w in WINDOWS_TO_RUN:
    ladder_seed_results[w].update(ladder_control_results[w])
    if INCLUDE_ALL_INDICATORS_STAGE:
        ladder_seed_results[w].update(all_indicators_results[w])
    for source in (native_results, cross_results):
        for dataset in ladder_labels:
            if dataset in source.get(w, {}):
                ladder_seed_results[w][dataset] = source[w][dataset]

ladder_metrics = make_metrics_table(
    combined_agg, WINDOWS_TO_RUN, ladder_labels, list(ALGOS),
    save=False, results=ladder_seed_results,
)

# Trades per (window, dataset, algo): 4 of the 6 stages (daily baseline, daily hybrid,
# scheduler hybrid, scheduler binary state) are native conditions dow_3 already scored
# -- pulled straight from trades_lookup (built above for the 2x2 table). The 2
# "restricted" stages are the genuinely new inference from the ladder-trades sweep
# cell above -- scored here from ladder_trades via the same CI formula.
from utils.tables import trades_ci_stats

ladder_trades_rows = []
for w in WINDOWS_TO_RUN:
    for dataset in ladder_labels:
        for algo in ALGOS:
            if dataset in ladder_trades.get(w, {}) and algo in ladder_trades[w][dataset]:
                row = {'window': w, 'dataset': dataset, 'algo': algo}
                row.update(trades_ci_stats(ladder_trades[w][dataset][algo]))
                ladder_trades_rows.append(row)
            elif (w, dataset, algo) in trades_lookup.index:
                row = {'window': w, 'dataset': dataset, 'algo': algo}
                row.update(trades_lookup.loc[(w, dataset, algo)].to_dict())
                ladder_trades_rows.append(row)
ladder_trades_df = pd.DataFrame(ladder_trades_rows)

# Keep both the plain (seed-averaged-curve) and _mean (per-seed-then-averaged) columns
# for every metric, not just Return% -- paired_delta_table below reads the _mean ones
# exclusively (see the aggregation comment in the "Merge native + cross results" cell);
# the plain ones stay here only for anyone browsing ladder_table row by row.
ladder_table = ladder_metrics.reset_index().merge(
    pd.DataFrame(stage_rows), on='dataset', how='left', validate='many_to_one',
).merge(
    ladder_trades_df, on=['window', 'dataset', 'algo'], how='left',
).loc[:, [
    'family', 'stage', 'window', 'dataset', 'algo', 'Return%', 'Sharpe', 'MaxDD%', 'Seeds',
    'Return%_mean', 'Return%_std', 'Return%_ci_lo', 'Return%_ci_hi',
    'Sharpe_mean',  'Sharpe_std',  'Sharpe_ci_lo',  'Sharpe_ci_hi',
    'MaxDD%_mean',  'MaxDD%_std',  'MaxDD%_ci_lo',  'MaxDD%_ci_hi',
    'Trades_mean',  'Trades_std',  'Trades_ci_lo',  'Trades_ci_hi',
]]
ladder_table['stage'] = pd.Categorical(
    ladder_table['stage'], categories=stage_order, ordered=True,
)
ladder_table = ladder_table.sort_values(['family', 'stage', 'window', 'algo'])

# Each delta uses only window--algorithm cells available for both compared stages.
# Pivoted on the _mean columns (per-seed metric, then averaged) -- NOT the plain
# Return%/Sharpe/MaxDD% columns, which come from averaging the 5 seed curves together
# before computing the metric once. That curve-averaging mechanically cancels cross-seed
# volatility (the way averaging independent portfolios does), which runs Sharpe
# optimistic and MaxDD shallow -- confirmed by the AGG_METHOD='mean_metrics' redo of the
# 2x2 table above, where every comparison that was reliable under the plain aggregation
# stayed reliable under the corrected one, just with slightly smaller point estimates.
paired = ladder_table.pivot_table(
    index=['family', 'window', 'algo'], columns='stage',
    values=['Return%_mean', 'Sharpe_mean', 'MaxDD%_mean'], aggfunc='first',
)
paired_deltas = []
for family, spec in SCHEDULER_FAMILIES.items():
    family_pairs = paired.loc[family]
    for stage in stage_order[1:]:
        if ('Return%_mean', stage) not in family_pairs.columns:
            continue
        base_return = family_pairs[('Return%_mean', 'daily baseline')]
        test_return = family_pairs[('Return%_mean', stage)]
        matched = base_return.notna() & test_return.notna()
        if not matched.any():
            continue
        paired_deltas.append({
            'family': family, 'comparison': f'{stage} minus daily baseline',
            'matched_cells': int(matched.sum()),
            'mean_return_delta_pp': (test_return[matched] - base_return[matched]).mean(),
            'mean_sharpe_delta': (
                family_pairs[('Sharpe_mean', stage)][matched]
                - family_pairs[('Sharpe_mean', 'daily baseline')][matched]
            ).mean(),
            'mean_maxdd_delta_pp': (
                family_pairs[('MaxDD%_mean', stage)][matched]
                - family_pairs[('MaxDD%_mean', 'daily baseline')][matched]
            ).mean(),
        })
paired_delta_table = pd.DataFrame(paired_deltas).round(3)

# Separate filenames when the random-placebo stage is included, so the two runs
# (with/without INCLUDE_RANDOM_PLACEBO_STAGE) never overwrite each other.
ladder_suffix = '_placebo' if INCLUDE_RANDOM_PLACEBO_STAGE else ''
ladder_out_dir = Path('tables_mv2boll/ladder')
ladder_out_dir.mkdir(parents=True, exist_ok=True)
try:
    ladder_table.to_csv(ladder_out_dir / f'scheduler_ladder_controls{ladder_suffix}.csv', index=False)
    paired_delta_table.to_csv(ladder_out_dir / f'scheduler_ladder_paired_deltas{ladder_suffix}.csv', index=False)
    print(f'Saved scheduler ladder tables under {ladder_out_dir}/ (suffix={ladder_suffix!r})')
except OSError as exc:
    print(f'Could not save scheduler ladder tables: {exc}')

display(ladder_table)
display(paired_delta_table)


Saved scheduler ladder tables under tables_mv2boll/ladder/ (suffix='_placebo')


/tmp/ipykernel_35875/1074185818.py:122: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  paired = ladder_table.pivot_table(


,family,stage,window,dataset,algo,Return%,Sharpe,MaxDD%,Seeds,Return%_mean,...,Sharpe_ci_lo,Sharpe_ci_hi,MaxDD%_mean,MaxDD%_std,MaxDD%_ci_lo,MaxDD%_ci_hi,Trades_mean,Trades_std,Trades_ci_lo,Trades_ci_hi
36,all,daily benchmark,w1,all,a2c,26.23,0.935,-26.24,5,26.232,...,0.628,1.233,-26.238,4.601,-31.951,-20.525,144.4,38.920,96.074,192.726
37,all,daily benchmark,w1,all,ppo,33.21,1.755,-9.18,5,33.210,...,1.380,2.001,-9.938,2.215,-12.688,-7.188,2314.8,621.130,1543.566,3086.034
38,all,daily benchmark,w1,all,sac,18.47,0.729,-27.11,5,18.472,...,0.517,0.914,-27.110,2.840,-30.636,-23.584,794.2,170.036,583.073,1005.327
138,all,daily benchmark,w3,all,a2c,12.42,0.792,-14.73,5,12.418,...,0.262,1.251,-15.596,1.694,-17.700,-13.492,66.8,9.935,54.464,79.136
139,all,daily benchmark,w3,all,ppo,21.19,1.222,-17.50,5,21.190,...,0.332,1.666,-17.670,5.747,-24.806,-10.534,1163.2,361.606,714.206,1612.194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,moving_average,daily benchmark restricted to random dates,w3,all__on_random_moving_average,ppo,20.47,1.410,-12.36,5,20.464,...,0.718,1.644,-12.824,5.957,-20.221,-5.427,895.0,227.129,612.982,1177.018
146,moving_average,daily benchmark restricted to random dates,w3,all__on_random_moving_average,sac,13.92,0.870,-17.87,5,13.922,...,0.491,1.162,-18.390,3.039,-22.164,-14.616,314.0,86.660,206.397,421.603
93,moving_average,daily benchmark restricted to random dates,w4,all__on_random_moving_average,a2c,-6.93,-0.264,-17.49,5,-6.928,...,-0.479,-0.014,-18.538,2.561,-21.718,-15.358,112.4,68.835,26.930,197.870
94,moving_average,daily benchmark restricted to random dates,w4,all__on_random_moving_average,ppo,-4.35,-0.141,-17.33,5,-4.352,...,-0.432,0.180,-18.592,2.372,-21.537,-15.647,1202.0,103.673,1073.274,1330.726


,family,comparison,matched_cells,mean_return_delta_pp,mean_sharpe_delta,mean_maxdd_delta_pp
0,moving_average,daily hybrid minus daily baseline,9,0.218,0.019,0.500
1,moving_average,baseline restricted to selected dates minus da...,9,1.865,0.143,2.808
2,moving_average,baseline restricted to random dates minus dail...,9,3.114,0.213,4.137
3,moving_average,scheduler hybrid minus daily baseline,9,2.502,0.185,3.138
4,moving_average,scheduler binary state minus daily baseline,9,1.952,0.135,3.084
5,moving_average,daily benchmark restricted to selected dates m...,9,1.590,0.110,2.603
6,moving_average,daily benchmark restricted to random dates min...,9,2.510,0.174,3.997
7,bollinger,daily hybrid minus daily baseline,9,0.933,0.052,0.530
8,bollinger,baseline restricted to selected dates minus da...,9,0.587,0.056,1.568
9,bollinger,baseline restricted to random dates minus dail...,9,2.747,0.130,2.544


In [28]:
# Scheduler-ladder seed-level table (scheduler_ladder_seed_controls_placebo.csv) --
# one row per (family, stage, window, algo, seed). The 5 native stages (daily
# baseline/hybrid, scheduler hybrid/binary, daily benchmark) come from dow_3's
# metrics_seeds_table.csv; the 4 restricted stages come from this notebook's own
# per-seed curves/trades (ladder_control_results/ladder_trades for the baseline
# restrictions, all_indicators_results/all_indicators_trades for the daily-benchmark
# restrictions, both already folded into ladder_seed_results/ladder_trades above).
# Reuses stage_rows (built above) so every stage/dataset pairing stays in sync with
# the aggregated table automatically -- restricted datasets are recognisable by the
# '__on_' label convention used throughout this notebook.
seed_rows_ladder = []
for row in stage_rows:
    family, stage, dataset = row['family'], row['stage'], row['dataset']
    is_restricted = '__on_' in dataset
    for w in WINDOWS_TO_RUN:
        for algo in ALGOS:
            if is_restricted:
                curves = ladder_seed_results[w].get(dataset, {}).get(algo, [])
                trades = ladder_trades.get(w, {}).get(dataset, {}).get(algo, [])
                if trades and len(trades) != len(curves):
                    print(f'  [WARN] {w}/{dataset}/{algo}: {len(curves)} curves vs {len(trades)} trades -- Trades left blank')
                    trades = []
                for i, curve in enumerate(curves):
                    seed = int(curve.name[1:]) if curve.name and curve.name.startswith('s') else SEEDS[i]
                    m = compute_metrics(curve)
                    seed_rows_ladder.append({'family': family, 'stage': stage, 'window': w, 'algo': algo,
                                              'seed': seed, 'Return%': m['Return%'], 'Sharpe': m['Sharpe'],
                                              'MaxDD%': m['MaxDD%'], 'Trades': trades[i] if trades else np.nan})
            else:
                if (w, dataset, algo) not in metrics_seeds_idx.index:
                    continue
                sub = metrics_seeds_idx.loc[[(w, dataset, algo)]]
                for _, r in sub.iterrows():
                    seed_rows_ladder.append({'family': family, 'stage': stage, 'window': w, 'algo': algo,
                                              'seed': int(r['seed']), 'Return%': r['Return%'], 'Sharpe': r['Sharpe'],
                                              'MaxDD%': r['MaxDD%'], 'Trades': r['Trades']})

df_ladder_seeds = pd.DataFrame(seed_rows_ladder)
ladder_seed_out_path = ladder_out_dir / f'scheduler_ladder_seed_controls{ladder_suffix}.csv'
df_ladder_seeds.to_csv(ladder_seed_out_path, index=False)
print(f'✓ Saved: {ladder_seed_out_path}  ({len(df_ladder_seeds)} rows)')


/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipyke

/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipyke

/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipykernel_35875/53139527.py:30: PerformanceWarning: indexing past lexsort depth may impact performance.
  if (w, dataset, algo) not in metrics_seeds_idx.index:
/tmp/ipyke

✓ Saved: tables_mv2boll/ladder/scheduler_ladder_seed_controls_placebo.csv  (765 rows)
